# WAXAL ASR — w2v-BERT 2.0 CTC

Joint model over Lingala / Luganda / Shona. Self-contained: every source file is
written to disk by the cells below, so there are no custom package dependencies.

**Phase 1 test labels are public and must not be used.** `waxal.data` refuses to
load the labeled test split; inference reads the test *audio* with the
transcription column dropped on load. Phase 1 leaderboard rank carries no signal —
watch the speaker-disjoint validation score instead.

Settings: **GPU T4 x2** (or P100), and **Internet ON** (the dataset streams from
Hugging Face).

## 1. Environment

In [ ]:
!pip install -q -U "transformers>=4.44" datasets jiwer accelerate soundfile librosa

import os, pathlib
# /kaggle/working is capped at ~20 GB and the labeled audio is ~12.6 GB; keep the
# HF cache on the larger scratch volume so extraction doesn't hit the wall.
os.environ["HF_HOME"] = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/hf/datasets"
pathlib.Path("/kaggle/temp/hf").mkdir(parents=True, exist_ok=True)

!df -h /kaggle/working /kaggle/temp | head -5
!nvidia-smi --query-gpu=name,memory.total --format=csv

Optional: a Hugging Face token avoids anonymous rate limits on the ~12.6 GB
download. Add it under **Add-ons → Secrets** as `HF_TOKEN`. The dataset is public,
so this only affects speed.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded")
except Exception as e:
    print(f"no HF token ({type(e).__name__}) — continuing anonymously")

## 2. Source

Generated from the repo — edit there, not here.

In [ ]:
%%writefile src/waxal/__init__.py

In [ ]:
%%writefile src/waxal/metric.py
"""Competition metric: 0.5 * WER + 0.5 * CER.

Zindi does not publish whether the metric is corpus-level (total edits / total
reference length) or the mean of per-utterance rates. They differ, sometimes by
a lot, so we compute both and treat the gap as a warning sign. Calibrate against
the public leaderboard once we have a scored submission.
"""

from dataclasses import dataclass

import jiwer


@dataclass
class Score:
    wer: float
    cer: float
    combined: float
    wer_mean: float
    cer_mean: float
    combined_mean: float

    def __str__(self) -> str:
        return (
            f"corpus: WER {self.wer:.4f}  CER {self.cer:.4f}  -> {self.combined:.4f}\n"
            f"mean:   WER {self.wer_mean:.4f}  CER {self.cer_mean:.4f}  -> {self.combined_mean:.4f}"
        )


def score(refs: list[str], hyps: list[str]) -> Score:
    if len(refs) != len(hyps):
        raise ValueError(f"length mismatch: {len(refs)} refs vs {len(hyps)} hyps")

    # jiwer errors on empty references; a blank hypothesis is legal (and is what
    # a model emits when it hears nothing), so only guard the reference side.
    pairs = [(r, h) for r, h in zip(refs, hyps) if r.strip()]
    if len(pairs) < len(refs):
        print(f"warning: dropped {len(refs) - len(pairs)} pairs with empty reference")
    r_ok, h_ok = [p[0] for p in pairs], [p[1] for p in pairs]

    wer = jiwer.wer(r_ok, h_ok)
    cer = jiwer.cer(r_ok, h_ok)
    per_wer = [jiwer.wer(r, h) for r, h in pairs]
    per_cer = [jiwer.cer(r, h) for r, h in pairs]
    wer_mean = sum(per_wer) / len(per_wer)
    cer_mean = sum(per_cer) / len(per_cer)

    return Score(
        wer=wer,
        cer=cer,
        combined=0.5 * wer + 0.5 * cer,
        wer_mean=wer_mean,
        cer_mean=cer_mean,
        combined_mean=0.5 * wer_mean + 0.5 * cer_mean,
    )


def score_by_language(refs: list[str], hyps: list[str], langs: list[str]) -> dict[str, Score]:
    """Per-language breakdown. The overall metric hides which language is dragging."""
    out = {}
    for lang in sorted(set(langs)):
        idx = [i for i, l in enumerate(langs) if l == lang]
        out[lang] = score([refs[i] for i in idx], [hyps[i] for i in idx])
    return out

In [ ]:
%%writefile src/waxal/normalize.py
"""Transcript cleanup for CTC training.

Deliberately conservative. We measured the cost of emitting normalized text
against the raw cased references (scripts/normalization_cost.py):

    lowercase + strip punctuation          -> 0.1011 combined
    ... plus deterministic recasing        -> 0.0703 combined
    raw cased + punctuated                 -> 0.0000

So case and punctuation stay. They're largely positional (89.3% of references
start with a capital, 83.8% end in terminal punctuation), which CTC learns
cheaply. All this module does is fold away the long tail of junk characters so
the CTC alphabet stays small and every symbol has enough training signal.
"""

import re
import unicodedata

# Rare characters that are almost certainly transcription noise or encoding
# damage, mapped to their intended form. Counts are from the 38k train rows.
CHAR_MAP = {
    "\xa0": " ", "​": "", "﻿": "",
    "«": '"', "»": '"', "“": '"', "”": '"', "„": '"',
    "‘": "'", "’": "'", "‛": "'", "`": "'", "´": "'",
    "–": "-", "—": "-", "‑": "-",
    "ᵑ": "ŋ",            # superscript n -> the real Luganda velar nasal
    "Ŋ": "ŋ",            # only a handful of uppercase forms; fold to lowercase
    "Ķ": "K", "ķ": "k", "Ĺ": "L", "ĺ": "l", "ĝ": "g", "ā": "a",
    "Œ": "OE", "œ": "oe", "þ": "th", "×": "x",
    "⭐": "", "️": "",     # emoji + variation selector
    "…": ".",
}

# The alphabet we actually train on. Anything outside this is dropped.
KEEP = set(
    "abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    "ŋ"                       # Luganda
    "àáâçèéêëìíîïòóôùúûü"      # Lingala/Shona diacritics (lowercase only)
    " '-.,!?;:()\""
)

_ACCENT_UPPER = str.maketrans("ÀÁÂÇÈÉÊËÌÍÎÏÒÓÔÙÚÛÜ", "àáâçèéêëìíîïòóôùúûü")


def clean(text: str) -> str:
    """Normalize a transcript to the training alphabet, preserving case/punctuation."""
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFC", text)
    for src, dst in CHAR_MAP.items():
        text = text.replace(src, dst)

    # Uppercase accented letters are vanishingly rare; folding them to lowercase
    # avoids spending alphabet slots on symbols with ~no training signal.
    text = text.translate(_ACCENT_UPPER)

    # Digits are read aloud as words, so a literal digit is never the right
    # target. Drop them rather than teach the model an unpronounceable symbol.
    text = re.sub(r"\d+", " ", text)

    text = "".join(c for c in text if c in KEEP)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def alphabet(texts: list[str]) -> list[str]:
    """The CTC vocabulary implied by a corpus, after cleaning."""
    seen = set()
    for t in texts:
        seen.update(clean(t))
    return sorted(seen)

In [ ]:
%%writefile src/waxal/hw.py
"""GPU capability checks.

torch.cuda.is_bf16_supported() returns True on a Tesla T4 (sm_75), which has no
native bf16 -- recent PyTorch counts *emulated* bf16 as supported. Training in
emulated bf16 is far slower than the fp16 tensor cores the card actually has, so
selecting precision from that check silently cripples throughput on pre-Ampere
hardware. Gate on compute capability instead: bf16 is native from sm_80 (Ampere).
"""

import torch


def supports_bf16() -> bool:
    """True only where bf16 runs on hardware, not through emulation."""
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability()
    return major >= 8


def describe() -> str:
    if not torch.cuda.is_available():
        return "cpu"
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability()
    native = supports_bf16()
    emulated = torch.cuda.is_bf16_supported() and not native
    note = " (bf16 emulated -- using fp16 instead)" if emulated else ""
    return f"{name} sm_{major}{minor} bf16_native={native}{note}"

In [ ]:
%%writefile src/waxal/data.py
"""Dataset assembly for the WAXAL ASR challenge.

Two things this module is careful about:

1. The HF `test` split is the Phase 1 test set and its labels are public. We
   never load it. Phase 1 rank is meaningless and using those labels is an
   explicit rules breach; Phase 2 decides the prizes.

2. Phase 2 ships no metadata at all -- no language tag, no speaker id. So the
   model must be language-agnostic at inference, and our validation split must
   be speaker-disjoint or it will flatter us exactly where Phase 2 will punish.
"""

from dataclasses import dataclass

import datasets
import numpy as np

LANGS = ("lin", "lug", "sna")
HF_REPO = "google/WaxalNLP"
SR = 16_000


# datasets >= 4.0 hands back a torchcodec AudioDecoder instead of the old
# {"array", "sampling_rate"} dict. Both shapes are handled here so the pipeline
# doesn't depend on which version the runtime happens to install.

def audio_duration(a) -> float | None:
    """Duration in seconds, without decoding the waveform where possible."""
    if isinstance(a, dict):
        arr, sr = a.get("array"), a.get("sampling_rate")
        if arr is not None and sr:
            return len(arr) / sr
        if a.get("num_samples") and sr:
            return a["num_samples"] / sr
        return None

    meta = getattr(a, "metadata", None)
    for attr in ("duration_seconds", "duration_seconds_from_header"):
        v = getattr(meta, attr, None)
        if v:
            return float(v)
    frames, sr = getattr(meta, "num_frames", None), getattr(meta, "sample_rate", None)
    if frames and sr:
        return frames / sr
    return None      # unknown -> caller keeps the clip rather than dropping it


def audio_array(a) -> tuple[np.ndarray, int]:
    """Decoded mono waveform and its sample rate."""
    if isinstance(a, dict):
        arr = np.asarray(a["array"], dtype=np.float32)
        sr = a["sampling_rate"]
    else:
        samples = a.get_all_samples()
        arr = samples.data.numpy()
        sr = int(samples.sample_rate)
    if arr.ndim > 1:
        arr = arr.mean(axis=0)      # fold any stereo down to mono
    return arr.astype(np.float32), sr


def _files(lang: str, split: str) -> str:
    return f"data/ASR/{lang}/{lang}-{split}-*.parquet"


def load_labeled(langs=LANGS, splits=("train", "validation"), num_proc: int = 4):
    """Load the labeled portion of the target languages. Never touches `test`."""
    if "test" in splits:
        raise ValueError(
            "the HF `test` split is the Phase 1 test set with public labels -- "
            "loading it risks contaminating training and breaches the rules"
        )
    parts = []
    for lang in langs:
        for split in splits:
            ds = datasets.load_dataset(
                HF_REPO, data_files={split: _files(lang, split)}, split=split,
                num_proc=num_proc,
            )
            parts.append(ds)
    ds = datasets.concatenate_datasets(parts)
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SR))


def load_test_audio(langs=LANGS, num_proc: int = 4):
    """Phase 1 test *audio only* -- the transcription column is dropped on load.

    Running our model over the Phase 1 test audio and submitting the predictions
    is legitimate: it validates the submission format and the inference path.
    What the rules forbid is using the public ground-truth labels. Dropping the
    column here means those labels never enter the process at all, so there is
    no path by which they could leak into a submission.
    """
    parts = []
    for lang in langs:
        ds = datasets.load_dataset(
            HF_REPO, data_files={"test": _files(lang, "test")}, split="test",
            num_proc=num_proc,
        )
        parts.append(ds.remove_columns([c for c in ds.column_names
                                        if c not in ("id", "audio")]))
    ds = datasets.concatenate_datasets(parts)
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SR))


def load_phase2_audio(path: str, num_proc: int = 4):
    """Phase 2 evaluation audio, whatever form it arrives in.

    Phase 2 ships no metadata, so this deliberately assumes nothing beyond an id
    and an audio payload. Adjust the loader once the actual format is published.
    """
    ds = datasets.load_dataset("audiofolder", data_dir=path, split="train",
                               num_proc=num_proc)
    return ds.cast_column("audio", datasets.Audio(sampling_rate=SR))


@dataclass
class Split:
    train: datasets.Dataset
    valid: datasets.Dataset

    def __str__(self) -> str:
        return f"train={len(self.train):,}  valid={len(self.valid):,}"


def speaker_disjoint_split(ds, valid_frac: float = 0.06, seed: int = 42) -> Split:
    """Hold out whole speakers, stratified by language.

    A random row-level split leaks speakers across the boundary: the model
    memorizes voices and validation reports a score the Phase 2 set will not
    reproduce. We hold out entire speakers per language instead, so validation
    measures generalization to unheard voices -- which is what Phase 2 is.
    """
    import random

    by_lang: dict[str, set[str]] = {}
    for lang, spk in zip(ds["language"], ds["speaker_id"]):
        by_lang.setdefault(lang, set()).add(spk)

    rng = random.Random(seed)
    held: set[tuple[str, str]] = set()
    for lang, spks in by_lang.items():
        spks = sorted(spks)
        rng.shuffle(spks)
        n = max(1, round(len(spks) * valid_frac))
        held.update((lang, s) for s in spks[:n])
        print(f"  {lang}: holding out {n}/{len(spks)} speakers")

    flags = [(l, s) in held for l, s in zip(ds["language"], ds["speaker_id"])]
    idx_v = [i for i, f in enumerate(flags) if f]
    idx_t = [i for i, f in enumerate(flags) if not f]
    return Split(train=ds.select(idx_t), valid=ds.select(idx_v))


def filter_usable(ds, min_s: float = 0.5, max_s: float = 30.0, min_chars: int = 3):
    """Drop rows that would waste compute or destabilize CTC.

    CTC requires the target to be no longer than the encoder output, so clips
    that are too short for their transcript produce inf loss. Over-long clips
    blow up memory quadratically in attention for a handful of examples.
    """
    from .normalize import clean

    def ok(row) -> bool:
        txt = clean(row["transcription"] or "")
        if len(txt) < min_chars:
            return False
        dur = audio_duration(row["audio"])
        if dur is None:
            return True      # can't tell cheaply; let the clip through
        if not (min_s <= dur <= max_s):
            return False
        # w2v-BERT downsamples ~2x per 10ms frame -> ~25 frames/sec of output.
        return len(txt) <= dur * 25

    return ds.filter(ok, desc="filtering usable clips")

In [ ]:
%%writefile scripts/train_ctc.py
"""Fine-tune w2v-BERT 2.0 with a CTC head on Lingala / Luganda / Shona.

Why CTC over the starter notebook's Gemma 3n + LoRA:
  * The metric is 50% CER. A generative decoder that hallucinates a fluent wrong
    sentence is catastrophic on both halves; CTC degrades into local character
    noise, which the CER half forgives.
  * Phase 2 provides no language tag. The three languages share <1% of their
    vocabulary, so one joint model over a shared alphabet infers the language
    from acoustics -- no LID stage, no metadata dependency.
  * CTC output can be rescored with an n-gram LM (pyctcdecode) for a cheap
    further gain. A generative model offers no equivalent.

Run:
    python scripts/train_ctc.py --output-dir out/ctc-v1
"""

import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import datasets
import numpy as np
import torch
import transformers

from waxal import data as wdata
from waxal import hw
from waxal.metric import score, score_by_language
from waxal.normalize import clean

MODEL_ID = "facebook/w2v-bert-2.0"


def build_tokenizer(texts: list[str], out_dir: Path) -> transformers.Wav2Vec2CTCTokenizer:
    chars = sorted({c for t in texts for c in clean(t)})
    # "|" stands in for space so the tokenizer can treat it as a normal symbol.
    vocab = {c if c != " " else "|": i for i, c in enumerate(chars)}
    vocab["[UNK]"] = len(vocab)
    vocab["[PAD]"] = len(vocab)          # doubles as the CTC blank
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "vocab.json").write_text(json.dumps(vocab, ensure_ascii=False, indent=1))
    print(f"vocab: {len(vocab)} symbols -> {out_dir/'vocab.json'}")
    return transformers.Wav2Vec2CTCTokenizer(
        str(out_dir / "vocab.json"),
        unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|",
    )


class Collator:
    """Pads audio features and labels independently; masks label padding to -100."""

    def __init__(self, processor):
        self.p = processor

    def __call__(self, features):
        audio = self.p.pad(
            [{"input_features": f["input_features"]} for f in features],
            padding=True, return_tensors="pt",
        )
        labels = self.p.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in features],
            padding=True, return_tensors="pt",
        )
        # CTC loss ignores -100, so padded label positions contribute nothing.
        audio["labels"] = labels["input_ids"].masked_fill(
            labels.attention_mask.ne(1), -100
        )
        return audio


class LengthGroupedSampler(torch.utils.data.Sampler):
    """Batch clips of similar duration together.

    transformers 5.x dropped `group_by_length`, and without it every batch is
    padded to its longest member -- with clips ranging from 0.5s to 30s that is a
    large fraction of the compute spent on padding. Shuffle, cut into megabatches,
    sort each by length, then shuffle the batch order: near-uniform batches while
    keeping enough randomness that the model doesn't see the same grouping twice.
    """

    def __init__(self, lengths: list[int], batch_size: int, seed: int = 42,
                 megabatch_mult: int = 50):
        self.lengths = lengths
        self.batch_size = batch_size
        self.megabatch_size = batch_size * megabatch_mult
        self.seed = seed
        self.epoch = 0

    def __len__(self) -> int:
        return len(self.lengths)

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch      # reshuffles grouping each epoch

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed + self.epoch)
        indices = torch.randperm(len(self.lengths), generator=g).tolist()

        megabatches = [indices[i:i + self.megabatch_size]
                       for i in range(0, len(indices), self.megabatch_size)]
        # Longest-first inside each megabatch surfaces the worst-case batch early,
        # so an OOM shows up in the first minute rather than an hour in.
        megabatches = [sorted(mb, key=lambda i: self.lengths[i], reverse=True)
                       for mb in megabatches]

        batches = [mb[i:i + self.batch_size]
                   for mb in megabatches
                   for i in range(0, len(mb), self.batch_size)]
        order = torch.randperm(len(batches), generator=g).tolist()
        for b in order:
            yield from batches[b]


class LengthGroupedTrainer(transformers.Trainer):
    """Trainer that uses LengthGroupedSampler when the dataset carries lengths."""

    def _get_train_sampler(self, *args, **kwargs):
        ds = self.train_dataset
        if ds is None or "length" not in getattr(ds, "column_names", []):
            return super()._get_train_sampler(*args, **kwargs)
        return LengthGroupedSampler(
            ds["length"], self.args.per_device_train_batch_size, self.args.seed
        )


def make_training_args(**kwargs) -> transformers.TrainingArguments:
    """Build TrainingArguments, dropping keys this transformers version rejects.

    The argument set churns across releases (v5 removed `group_by_length`, v4.46
    renamed `evaluation_strategy` to `eval_strategy`). Kaggle, Colab and RunPod
    will not agree on a version, so filter against the actual signature rather
    than pinning -- and say out loud what got dropped, since a silently ignored
    argument is how you end up wondering why training is slow.
    """
    import inspect

    sig = inspect.signature(transformers.TrainingArguments.__init__)
    accepted = set(sig.parameters)

    # Renames, newest name first: try each until one is accepted.
    aliases = {
        "eval_strategy": ("eval_strategy", "evaluation_strategy"),
        "warmup_ratio": ("warmup_ratio",),
    }
    resolved, dropped = {}, []
    for key, value in kwargs.items():
        for name in aliases.get(key, (key,)):
            if name in accepted:
                resolved[name] = value
                break
        else:
            dropped.append(key)

    if dropped:
        print(f"note: transformers {transformers.__version__} does not accept "
              f"{dropped} -- proceeding without")
        if "group_by_length" in dropped:
            print("      (length grouping off: more padding waste, same accuracy)")
    return transformers.TrainingArguments(**resolved)


def _resample(arr: np.ndarray, factor: float) -> np.ndarray:
    """Speed up (factor>1) or slow down the waveform, shifting pitch with it.

    Resampling without correcting pitch is deliberate -- it is the classic Kaldi
    speed-perturb recipe, and the pitch shift is what makes it act like extra
    speakers rather than just extra tempo. That is what Phase 2 tests.
    """
    if factor == 1.0:
        return arr
    n = int(round(len(arr) / factor))
    # Linear interpolation onto the resampled grid; good enough for augmentation
    # and avoids a scipy/librosa dependency in the hot path.
    idx = np.linspace(0, len(arr) - 1, n)
    return np.interp(idx, np.arange(len(arr)), arr).astype(np.float32)


def prepare(ds, processor, num_proc: int, speeds: tuple[float, ...] = (1.0,)):
    def fn(batch):
        # batched=True with batch_size=1: every field arrives as a 1-element list.
        arr, sr = wdata.audio_array(batch["audio"][0])
        labels = processor.tokenizer(clean(batch["transcription"][0])).input_ids
        feats, lens = [], []
        for f in speeds:
            x = processor(_resample(arr, f), sampling_rate=sr).input_features[0]
            feats.append(x)
            lens.append(len(x))
        return {
            "input_features": feats,
            "labels": [labels] * len(speeds),      # transcript is unchanged
            "language": [batch["language"][0]] * len(speeds),
            "length": lens,                        # drives LengthGroupedSampler
        }

    # batched with a 1-row batch: lets each input emit len(speeds) output rows.
    return ds.map(fn, remove_columns=ds.column_names, num_proc=num_proc,
                  batched=True, batch_size=1,
                  desc=f"extracting features (speeds={list(speeds)})")


def prepare_cached(ds, processor, num_proc: int, cache_dir: Path | None, tag: str,
                   key: dict, speeds: tuple[float, ...] = (1.0,)):
    """Feature extraction with an explicit on-disk cache.

    datasets.map caches by fingerprint, but the fingerprint hashes the mapped
    function *and its closure* -- which here includes the processor. Any edit to
    this file, or an unstable hash of the processor, silently invalidates it and
    you pay full extraction again. At ~40 minutes a run that is too expensive to
    leave to chance, so we save explicitly and validate against the parameters
    that actually affect the output.
    """
    if cache_dir is None:
        return prepare(ds, processor, num_proc, speeds)

    path = cache_dir / tag
    manifest = cache_dir / f"{tag}.json"
    if path.exists() and manifest.exists():
        stored = json.loads(manifest.read_text())
        if stored == key:
            print(f"reusing cached features: {path}")
            return datasets.load_from_disk(str(path))
        # Stale cache is worse than none -- it trains on the wrong thing silently.
        print(f"cache at {path} was built with different settings, rebuilding")
        print(f"  cached: {stored}\n  wanted: {key}")

    out = prepare(ds, processor, num_proc, speeds)
    path.parent.mkdir(parents=True, exist_ok=True)
    out.save_to_disk(str(path))
    manifest.write_text(json.dumps(key, indent=1, sort_keys=True))
    print(f"cached features -> {path}")
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--output-dir", type=Path, required=True)
    ap.add_argument("--epochs", type=float, default=6.0)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--grad-accum", type=int, default=4)
    ap.add_argument("--lr", type=float, default=5e-5)
    ap.add_argument("--num-proc", type=int, default=8)
    ap.add_argument("--valid-frac", type=float, default=0.06)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--limit", type=int, default=0, help="debug: cap rows loaded")
    ap.add_argument("--max-s", type=float, default=30.0,
                    help="drop clips longer than this. Attention is quadratic in "
                         "length, so a few long clips cost disproportionately")
    ap.add_argument("--min-s", type=float, default=0.5)
    ap.add_argument("--cache-dir", type=Path, default=None,
                    help="persist extracted features here so a failed run doesn't "
                         "repeat the ~40min extraction; put it on a persistent volume")
    ap.add_argument("--save-strategy", choices=("steps", "epoch"), default="epoch",
                    help="epoch: one checkpoint per epoch (default)")
    ap.add_argument("--save-steps", type=int, default=500,
                    help="only used with --save-strategy steps")
    ap.add_argument("--push-to-hub", type=str, default="",
                    help="HF repo id, e.g. ngam/waxal-ctc-v1. Checkpoints upload "
                         "as they are saved, surviving pod loss")
    ap.add_argument("--hub-public", action="store_true",
                    help="publish the Hub repo publicly. Off by default: the rules "
                         "forbid sharing work outside your team during the challenge")
    ap.add_argument("--resume", action="store_true",
                    help="resume from the last checkpoint in --output-dir")
    # SpecAugment. Applied inside the model at training time only, so it costs
    # nothing extra and does NOT invalidate the feature cache -- tune these freely.
    ap.add_argument("--mask-time-prob", type=float, default=0.05,
                    help="fraction of time steps masked (0.05-0.10 typical)")
    ap.add_argument("--mask-time-length", type=int, default=10)
    ap.add_argument("--mask-feature-prob", type=float, default=0.0,
                    help="frequency masking; 0.01-0.05 helps on unseen speakers")
    ap.add_argument("--mask-feature-length", type=int, default=64)
    # Speed perturbation. Changes the audio, so it DOES invalidate the cache and
    # multiplies extraction time and disk by len(factors).
    ap.add_argument("--speed-perturb", type=str, default="",
                    help="comma-separated factors, e.g. 0.9,1.0,1.1 -- triples the "
                         "training set with faster/slower copies")
    args = ap.parse_args()

    transformers.set_seed(args.seed)          # rules require reproducibility

    print("loading labeled data (train+validation only; test is off-limits)")
    ds = wdata.load_labeled(num_proc=args.num_proc)
    if args.limit:
        ds = ds.select(range(min(args.limit, len(ds))))
    print(f"  {len(ds):,} rows")

    ds = wdata.filter_usable(ds, min_s=args.min_s, max_s=args.max_s)
    print(f"  {len(ds):,} usable (clips {args.min_s}-{args.max_s}s)")

    print("building speaker-disjoint validation split")
    split = wdata.speaker_disjoint_split(ds, args.valid_frac, args.seed)
    print(f"  {split}")

    tokenizer = build_tokenizer(split.train["transcription"], args.output_dir)
    fe = transformers.AutoFeatureExtractor.from_pretrained(MODEL_ID)
    processor = transformers.Wav2Vec2BertProcessor(feature_extractor=fe, tokenizer=tokenizer)

    # Everything that changes the extracted features or which rows they cover.
    # The learning rate and epoch count deliberately aren't here -- they don't
    # affect features, so re-tuning them should reuse the cache.
    speeds = tuple(float(s) for s in args.speed_perturb.split(",")) \
        if args.speed_perturb else (1.0,)

    key = {
        "model": MODEL_ID, "langs": list(wdata.LANGS), "sr": wdata.SR,
        "valid_frac": args.valid_frac, "seed": args.seed, "limit": args.limit,
        "min_s": args.min_s, "max_s": args.max_s,   # change which rows survive
        "vocab": sorted(processor.tokenizer.get_vocab()),
    }
    # Augment training only. A perturbed validation set would measure the model
    # on audio the Phase 2 set will never contain, and would not be comparable
    # to the un-augmented baseline.
    train_ds = prepare_cached(split.train, processor, args.num_proc,
                              args.cache_dir, "train",
                              {**key, "speeds": list(speeds)}, speeds)
    valid_ds = prepare_cached(split.valid, processor, args.num_proc,
                              args.cache_dir, "valid", key)
    if speeds != (1.0,):
        print(f"speed perturbation {list(speeds)}: train {len(split.train):,} "
              f"-> {len(train_ds):,} rows")
    valid_langs = valid_ds["language"]
    valid_refs = [clean(t) for t in split.valid["transcription"]]

    model = transformers.Wav2Vec2BertForCTC.from_pretrained(
        MODEL_ID,
        attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0,
        # SpecAugment-style masking is the main regularizer here; the labeled
        # set is small relative to the 580M encoder.
        mask_time_prob=args.mask_time_prob,
        mask_time_length=args.mask_time_length,
        mask_feature_prob=args.mask_feature_prob,
        mask_feature_length=args.mask_feature_length,
        layerdrop=0.0,
        ctc_loss_reduction="mean",
        add_adapter=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
    )

    def compute_metrics(pred):
        ids = np.argmax(pred.predictions, axis=-1)
        hyps = processor.batch_decode(ids)
        s = score(valid_refs, hyps)
        per = score_by_language(valid_refs, hyps, valid_langs)
        out = {"wer": s.wer, "cer": s.cer, "combined": s.combined}
        out.update({f"combined_{l}": v.combined for l, v in per.items()})
        return out

    targs = make_training_args(
        output_dir=str(args.output_dir),
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_ratio=0.1,
        lr_scheduler_type="linear",
        # NOT torch.cuda.is_bf16_supported(): that reports True on a T4, where
        # bf16 is emulated and much slower than the card's fp16 tensor cores.
        bf16=hw.supports_bf16(),
        fp16=not hw.supports_bf16(),
        gradient_checkpointing=True,
        # Evaluate and save on the same cadence so load_best_model_at_end always
        # has a metric for every checkpoint it might pick.
        eval_strategy=args.save_strategy,
        save_strategy=args.save_strategy,
        **({"eval_steps": args.save_steps, "save_steps": args.save_steps}
           if args.save_strategy == "steps" else {}),
        logging_steps=50,
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model="combined",
        greater_is_better=False,          # lower WER/CER wins
        # We group by length ourselves (LengthGroupedTrainer); keep the extra
        # columns it needs, since the collator already selects what the model sees.
        remove_unused_columns=False,
        dataloader_num_workers=4,
        seed=args.seed,
        report_to=[],
        # "every_save" uploads each checkpoint as it is written, so a pod that
        # dies mid-run costs one epoch rather than the whole run.
        **({"push_to_hub": True,
            "hub_model_id": args.push_to_hub,
            "hub_strategy": "every_save",
            "hub_private_repo": not args.hub_public} if args.push_to_hub else {}),
    )

    trainer = LengthGroupedTrainer(
        model=model, args=targs,
        train_dataset=train_ds, eval_dataset=valid_ds,
        data_collator=Collator(processor),
        compute_metrics=compute_metrics,
    )
    resume = None
    if args.resume:
        last = transformers.trainer_utils.get_last_checkpoint(str(args.output_dir))
        if last:
            print(f"resuming from {last}")
            resume = last
        else:
            print(f"--resume given but no checkpoint in {args.output_dir}; "
                  "starting fresh")

    trainer.train(resume_from_checkpoint=resume)

    best = args.output_dir / "best"
    trainer.save_model(str(best))
    processor.save_pretrained(str(best))
    print(f"saved -> {best}")

    if args.push_to_hub:
        # The per-checkpoint uploads carry model weights but not the processor;
        # without it the repo can't tokenize or extract features on its own.
        processor.push_to_hub(args.push_to_hub, private=not args.hub_public)
        trainer.push_to_hub(commit_message="final best checkpoint")
        print(f"pushed -> https://huggingface.co/{args.push_to_hub}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/infer.py
"""Run a trained CTC model over evaluation audio and write a Zindi submission.

Phase 1 mode reads the public test audio but never its labels (see
waxal.data.load_test_audio). Phase 2 mode points at whatever directory the
organizers publish.

    python scripts/infer.py --model out/ctc-v1/best --phase 1 --out sub.csv
"""

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import pandas as pd
import torch
import transformers

from waxal import data as wdata


@torch.no_grad()
def transcribe(ds, model, processor, device, batch_size: int = 8) -> dict[str, str]:
    model.eval().to(device)
    out: dict[str, str] = {}
    for start in range(0, len(ds), batch_size):
        rows = ds[start:start + batch_size]
        arrays = [wdata.audio_array(a)[0] for a in rows["audio"]]
        feats = processor(
            arrays, sampling_rate=wdata.SR, return_tensors="pt", padding=True,
        ).to(device)
        with torch.autocast(device_type=device.type,
                            dtype=torch.bfloat16, enabled=device.type == "cuda"):
            logits = model(**feats).logits
        hyps = processor.batch_decode(logits.argmax(-1).cpu().numpy())
        out.update(zip(rows["id"], hyps))
        if start % (batch_size * 25) == 0:
            print(f"  {min(start + batch_size, len(ds)):,}/{len(ds):,}", flush=True)
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=Path, required=True)
    ap.add_argument("--phase", type=int, choices=(1, 2), default=1)
    ap.add_argument("--phase2-dir", type=str, default="")
    ap.add_argument("--sample-submission", type=Path,
                    default=Path("data/raw/SampleSubmission.csv"))
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--num-proc", type=int, default=4)
    args = ap.parse_args()

    processor = transformers.Wav2Vec2BertProcessor.from_pretrained(str(args.model))
    model = transformers.Wav2Vec2BertForCTC.from_pretrained(str(args.model))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if args.phase == 1:
        ds = wdata.load_test_audio(num_proc=args.num_proc)
    else:
        if not args.phase2_dir:
            ap.error("--phase 2 requires --phase2-dir")
        ds = wdata.load_phase2_audio(args.phase2_dir, num_proc=args.num_proc)
    print(f"transcribing {len(ds):,} clips on {device}")

    preds = transcribe(ds, model, processor, device, args.batch_size)

    sub = pd.read_csv(args.sample_submission, escapechar="\\")
    sub["Target"] = sub.ID.map(preds)

    missing = sub.Target.isna().sum()
    if missing:
        # A blank target still scores (badly) -- a missing row may not score at all.
        print(f"WARNING: {missing} ids had no prediction; filling with empty string")
        sub["Target"] = sub.Target.fillna("")

    empty = (sub.Target.str.strip() == "").sum()
    print(f"empty predictions: {empty:,}/{len(sub):,}")

    args.out.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(args.out, index=False)
    print(f"wrote {args.out}  ({len(sub):,} rows)")
    print(sub.head(3).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/bench.py
"""Find out where training time actually goes before renting a bigger GPU.

Observed 106 s/it on a T4 is far off what the arithmetic predicts, so the
bottleneck may not be compute at all. This times the two halves separately:

  * GPU: forward+backward on synthetic tensors, no data pipeline involved.
  * Data: iterating the real dataloader, no model involved.

If GPU time dominates, a faster card fixes it. If data time dominates, a faster
card changes nothing and the fix is in the pipeline.

    python scripts/bench.py --cache-dir /workspace/cache --batch-size 4
"""

import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import datasets
import torch
import transformers

from waxal import hw

MODEL_ID = "facebook/w2v-bert-2.0"


def bench_gpu(batch_size: int, frames: int, vocab: int, steps: int,
              checkpointing: bool, dtype: torch.dtype) -> float:
    """Seconds per optimizer step, synthetic data, nothing else in the way."""
    dev = torch.device("cuda")
    model = transformers.Wav2Vec2BertForCTC.from_pretrained(
        MODEL_ID, vocab_size=vocab, ctc_loss_reduction="mean",
        add_adapter=True, pad_token_id=vocab - 1,
    ).to(dev)
    if checkpointing:
        model.gradient_checkpointing_enable()
    model.train()

    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    amp = dtype
    scaler = torch.amp.GradScaler(enabled=amp == torch.float16)

    x = torch.randn(batch_size, frames, 160, device=dev)
    y = torch.randint(0, vocab - 2, (batch_size, frames // 12), device=dev)

    for i in range(steps + 2):          # first two warm up cudnn/allocator
        if i == 2:
            torch.cuda.synchronize()
            t0 = time.perf_counter()
        with torch.autocast("cuda", dtype=amp):
            loss = model(input_features=x, labels=y).loss
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / steps


def bench_data(cache_dir: Path, batch_size: int, workers: int, steps: int) -> tuple:
    """Seconds per batch pulled through the real dataloader, model excluded."""
    ds = datasets.load_from_disk(str(cache_dir / "train"))
    lengths = ds["length"]

    def collate(feats):
        import numpy as np
        m = max(len(f["input_features"]) for f in feats)
        out = np.zeros((len(feats), m, 160), dtype=np.float32)
        for i, f in enumerate(feats):
            a = np.asarray(f["input_features"], dtype=np.float32)
            out[i, :len(a)] = a
        return torch.from_numpy(out)

    dl = torch.utils.data.DataLoader(
        ds, batch_size=batch_size, collate_fn=collate,
        num_workers=workers, shuffle=True,
    )
    it = iter(dl)
    next(it)                              # warm up workers
    t0 = time.perf_counter()
    n = 0
    for _ in range(steps):
        try:
            next(it)
        except StopIteration:
            break
        n += 1
    return (time.perf_counter() - t0) / max(n, 1), lengths


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache-dir", type=Path, default=None)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--grad-accum", type=int, default=8)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--steps", type=int, default=5)
    ap.add_argument("--vocab", type=int, default=85)
    args = ap.parse_args()

    if not torch.cuda.is_available():
        print("no GPU visible")
        return
    print(f"GPU: {hw.describe()}\n")

    have_cache = args.cache_dir and (args.cache_dir / "train").exists()
    if args.cache_dir and not have_cache:
        print(f"no feature cache at {args.cache_dir/'train'} -- "
              "running the GPU benchmark only\n")
    if have_cache:
        per_batch, lengths = bench_data(args.cache_dir, args.batch_size,
                                        args.workers, 20)
        import statistics
        print("clip length in frames (100 frames ~= 1s of audio):")
        print(f"  mean {statistics.mean(lengths):.0f}  median "
              f"{statistics.median(lengths):.0f}  "
              f"p95 {sorted(lengths)[int(len(lengths)*0.95)]}  max {max(lengths)}")
        data_step = per_batch * args.grad_accum
        print(f"\ndata only: {per_batch:.3f}s/batch -> {data_step:.1f}s per "
              f"optimizer step ({args.grad_accum} accum)\n")
        frames = int(statistics.mean(lengths))
    else:
        data_step, frames = 0.0, 750
        print("(no --cache-dir: skipping data benchmark)\n")

    total = 0.0
    for dtype, label in ((torch.float16, "fp16"), (torch.bfloat16, "bf16")):
        for ckpt in (True, False):
            try:
                s = bench_gpu(args.batch_size, frames, args.vocab, args.steps,
                              ckpt, dtype)
                total = s * args.grad_accum
                print(f"gpu {label} checkpointing={str(ckpt):5s}: "
                      f"{s:.3f}s/microbatch -> {total:.1f}s per optimizer step")
            except torch.OutOfMemoryError:
                # Expected without checkpointing on a small card; not a failure.
                print(f"gpu {label} checkpointing={str(ckpt):5s}: OOM at "
                      f"batch {args.batch_size} x {frames} frames")
            finally:
                torch.cuda.empty_cache()

    if data_step:
        print(f"\nverdict: data {data_step:.1f}s vs gpu {total:.1f}s per step -> "
              f"{'DATA-BOUND (a faster GPU will not help)' if data_step > total else 'GPU-BOUND (a faster card helps)'}")


if __name__ == "__main__":
    main()

In [ ]:
import sys
sys.path.insert(0, "src")
from waxal.normalize import clean
from waxal.metric import score
assert clean("  Ndaba «x» 12 ⭐️  ") == 'Ndaba "x"'
print("modules OK")

## 3. Smoke test

A few hundred rows end-to-end first. The full run costs hours; a typo shouldn't
cost you one of them. Expect a *terrible* score here — 200 rows trains nothing.
What matters is that it completes without raising.

In [ ]:
!python scripts/train_ctc.py \
    --output-dir /kaggle/temp/smoke \
    --limit 200 --epochs 1 --batch-size 2 --grad-accum 1 \
    --valid-frac 0.25 --num-proc 2

## 4. Full training run

Kaggle sessions are capped at 12 hours (9 for GPU) and the weekly GPU quota is 30
hours, so this is sized to fit one session rather than to be optimal — treat it as
a baseline to beat on RunPod, not the final model.

Sized for a **T4 (15 GB, no bf16)**: batch 4 with 8 accumulation steps holds the
effective batch at 32 while keeping activations inside memory. A 580M model in
mixed precision spends ~10 GB on weights, the fp32 master copy, and AdamW moments
before a single activation is stored.

`CUDA_VISIBLE_DEVICES=0` pins this to one GPU on purpose. With both visible, the
Trainer silently switches to DataParallel, which changes the effective batch size
and adds a failure mode to debug on the first real run. Drop the prefix to use
both once a single-GPU run is known good.

`group_by_length` matters here: these clips vary a lot in duration, and batching
similar lengths together cuts padding waste substantially.

In [ ]:
# --cache-dir keeps the ~40min feature extraction across retries. /kaggle/temp
# dies with the session but /kaggle/working is capped at 20GB, and features
# (~11GB) plus checkpoints (~7GB) would leave no headroom there.
!CUDA_VISIBLE_DEVICES=0 python scripts/train_ctc.py \
    --output-dir /kaggle/working/ctc-v1 \
    --cache-dir /kaggle/temp/features \
    --epochs 3 --batch-size 4 --grad-accum 8 --lr 5e-5 \
    --num-proc 4 --valid-frac 0.06 --seed 42

**If loss goes `nan` and stays there:** that's fp16 underflow in the CTC loss, not
a data problem. T4 can't do bf16, so the fixes are to lower the learning rate to
`3e-5`, or raise warmup to `--warmup-ratio 0.2`. If it persists, training in fp32
works but roughly halves throughput.

**If it OOMs:** drop to `--batch-size 2 --grad-accum 16`.

## 4b. Where is the time going?

Run this **with training stopped** — it needs the GPU to itself. It times
forward+backward on synthetic tensors and, if a feature cache exists, the real
dataloader, then says which one dominates.

A faster GPU only helps the first number. If the dataloader dominates, renting an
A100 buys nothing.

In [ ]:
!python scripts/bench.py --batch-size 4 --grad-accum 8 \
    --cache-dir /kaggle/temp/features

## 5. Validation

The honest number. `combined` is the competition metric on **held-out speakers**;
the per-language breakdown shows which language is dragging — Luganda has the
least data (5,455 rows vs ~14k each for the others), so expect it to lag.

In [ ]:
import json, pathlib
state = json.loads(pathlib.Path("/kaggle/working/ctc-v1/best/trainer_state.json").read_text()) \
    if pathlib.Path("/kaggle/working/ctc-v1/best/trainer_state.json").exists() else None
if state:
    rows = [h for h in state["log_history"] if "eval_combined" in h]
    for h in rows[-5:]:
        per = {k.replace("eval_combined_", ""): round(v, 4)
               for k, v in h.items() if k.startswith("eval_combined_")}
        print(f"step {h['step']:>6}  combined {h['eval_combined']:.4f}  "
              f"wer {h['eval_wer']:.4f}  cer {h['eval_cer']:.4f}  {per}")
else:
    print("no trainer_state.json — run training first")

## 6. Submission

Phase 1 predictions — for format validation and pipeline confidence only. The
leaderboard score it returns is not meaningful, since others may be submitting
lookups against the public labels.

When Phase 2 lands (~26 July), switch to `--phase 2 --phase2-dir <path>`.

In [ ]:
!python scripts/infer.py \
    --model /kaggle/working/ctc-v1/best \
    --phase 1 \
    --sample-submission /kaggle/input/waxal-csvs/SampleSubmission.csv \
    --out /kaggle/working/submission.csv \
    --batch-size 8

In [ ]:
import pandas as pd
sub = pd.read_csv("/kaggle/working/submission.csv", escapechar="\\")
sample = pd.read_csv("/kaggle/input/waxal-csvs/SampleSubmission.csv", escapechar="\\")
assert list(sub.columns) == ["ID", "Target"], sub.columns
assert len(sub) == len(sample), (len(sub), len(sample))
assert sub.ID.tolist() == sample.ID.tolist(), "ID order must match SampleSubmission"
print(f"submission valid — {len(sub):,} rows")
sub.head()